#  LLM 성능평가 - Pairwise (비교 평가)

1. OpenEvals를 사용한 A/B 테스트 수행 방법을 이해한다
2. Reference-free와 Reference-based 비교 평가의 차이를 이해하고 적용한다
3. 이진(A/B/TIE) 평가와 점수 척도 평가를 모두 구현한다
4. Langfuse를 통해 평가 결과를 모니터링한다

---

## 환경 설정 및 준비

`(1) Env 환경변수`

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

# 필수 환경 변수 확인
required_vars = ["OPENAI_API_KEY", "GOOGLE_API_KEY", "LANGFUSE_SECRET_KEY"]
missing = [var for var in required_vars if not os.getenv(var)]
if missing:
    print(f"⚠️ 누락된 환경 변수: {', '.join(missing)}")
else:
    print("✅ 모든 환경 변수가 설정되었습니다.")

✅ 모든 환경 변수가 설정되었습니다.


`(2) 기본 라이브러리`

In [2]:
from glob import glob
from pprint import pprint
import json

`(3) langfuse handler 설정`

In [3]:
from langfuse.langchain import CallbackHandler

# LangChain 콜백 핸들러 생성
langfuse_handler = CallbackHandler()

`(4) Test Data`

In [4]:
# Test 데이터셋에 대한 QA 생성 결과를 리뷰한 후 다시 로드
import pandas as pd

df_qa_test = pd.read_excel("data/testset.xlsx")

print(f"테스트셋: {df_qa_test.shape[0]}개 문서")
df_qa_test.head(2)

테스트셋: 49개 문서


,user_input,reference_contexts,reference,synthesizer_name
0,"Tesla, Inc.는 미국에서 어떤 역할을 하고 있으며, 이 회사의 주요 제품과 ...","['Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 이 회...","Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사로, 전기 자동차(...",single_hop_specifc_query_synthesizer
1,Forbes Global 2000에서 테슬라 순위 뭐야?,['Tesla의 차량 생산은 2008년 Roadster로 시작하여 Model S (...,테슬라는 Forbes Global 2000에서 69위에 랭크되었습니다.,single_hop_specifc_query_synthesizer


---

## **LLM 애플리케이션 평가**

- **AI 평가**는 데이터셋, 평가자, 평가 방법론 세 가지 핵심 요소로 구성되며, 초기에는 **10-20개의 고품질 예제**로 시작하는 것이 효과적

- 평가 방식은 **인간 평가**와 **자동화 평가** 두 트랙으로 진행되며, 주관적 판단이 필요한 초기에는 인간 평가를, 확장이 필요한 경우 휴리스틱 기반 자동화 평가를 활용

- 평가는 **오프라인**과 **온라인** 환경에서 수행되며, 벤치마킹, 테스트, 실시간 모니터링 등 상황에 맞는 방법론을 적용

- 지속적인 **CI/CD 통합**과 모니터링 시스템 구축을 통해 평가 프로세스의 효율성과 신뢰성을 확보해야 함

### 1) **벡터스토어** 로드

- **Chroma DB** 설정에서 모델, 컬렉션명, 저장 경로 지정

In [5]:
# 벡터 저장소 로드
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

chroma_db = Chroma(
    collection_name="db_korean_cosine_metadata",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)

print(chroma_db._collection.count())

e:\sw\dev\ai\modu_llm7\etf-bot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


39


In [6]:
# 벡터저장소 검색기 생성
chroma_k = chroma_db.as_retriever(
    search_kwargs={'k': 4},
)

# 벡터저장소 검색기를 사용하여 검색
query = "Elon Musk는 Tesla의 초기 자금 조달과 경영 변화에 어떻게 관여했으며, 그 과정에서 어떤 논란에 직면했나요?"

retrieved_docs = chroma_k.invoke(query)

# 검색 결과 출력
for doc in retrieved_docs:
    print(f"- {doc.page_content} [출처: {doc.metadata['source']}]")
    print("-"*200)
    print()

- [출처] 이 문서는 테슬라(Tesla)에 대한 문서입니다.
----------------------------------
Tesla는 내부 고발자 보복, 근로자 권리 침해, 안전 결함, 홍보 부족, Musk의 논란의 여지가 있는 발언과 관련된 소송, 정부 조사 및 비판에 직면했습니다.

## 역사

### 창립 (2003–2004)

Tesla Motors, Inc.는 2003년 7월 1일에 Martin Eberhard와 Marc Tarpenning에 의해 설립되었으며, 각각 CEO와 CFO를 역임했습니다. Ian Wright는 얼마 지나지 않아 합류했습니다. 2004년 2월, Elon Musk는 750만 달러의 시리즈 A 자금 조달을 주도하여 회장 겸 최대 주주가 되었습니다. J. B. Straubel은 2004년 5월 CTO로 합류했습니다. 다섯 명 모두 공동 설립자로 인정받고 있습니다.

### Roadster (2005–2009) [출처: data\Tesla_KR.md]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

- [출처] 이 문서는 테슬라(Tesla)에 대한 문서입니다.
----------------------------------
## 파트너

Tesla는 Panasonic과 파트너십을 맺고 있으며 리튬 공급에 대한 장기 계약을 맺고 있습니다. 이전 파트너로는 Daimler와 Toyota가 있습니다.

## 소송 및 논란

Tesla는 성희롱, 노동 분쟁, 사기 혐의, 대리점 분쟁, 지적 재산권, 환경 위반, 재산 피해, 인종 차별, COVID-19 팬데믹 대응 및 수리 권리와 관련된 소송 및 논란에 직면했습니다.

## 비판

T

### 2) **BM25 검색기** 준비

- **BM25 검색기** 구현으로 문서 유사도 기반 검색 가능

- **한국어 텍스트 처리**를 위한 **Kiwi 토크나이저** 설정

- 참고: https://github.com/bab2min/kiwipiepy

In [7]:
# korean_docs 파일을 로드 (jsonlines 파일)
def load_jsonlines(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        docs = [json.loads(line) for line in f]
    return docs

korean_docs = load_jsonlines('korean_docs_final.jsonl')
print(f"로드된 문서: {len(korean_docs)}개")
pprint(korean_docs[0])

로드된 문서: 39개
('{"id":null,"metadata":{"source":"data/테슬라_KR.md","company":"테슬라","language":"ko"},"page_content":"<Document>\\nTesla, '
 'Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 이 회사는 전기 자동차(BEV), 고정형 배터리 에너지 저장 장치, 태양 '
 '전지판, 태양광 지붕널 및 관련 제품/서비스를 설계, 제조 및 판매합니다. 2003년 7월 Martin Eberhard와 Marc '
 'Tarpenning이 Tesla Motors로 설립했으며, Nikola Tesla를 기리기 위해 명명되었습니다. Elon Musk는 '
 '2004년 Tesla의 초기 자금 조달을 주도하여 2008년에 회장 겸 CEO가 '
 "되었습니다.\\n</Document>\\n<Source>이 문서는 미국 전기차 회사인 '테슬라'에 대한 "
 '문서입니다.</Source>","type":"Document"}')


In [8]:
# BM25 검색기 생성을 위해 문서 객체를 로드
documents = chroma_db.get()["documents"]
metadatas = chroma_db.get()["metadatas"]

# Document 객체로 변환
from langchain_core.documents import Document
docs = [Document(page_content=content, metadata=meta) for content, meta in zip(documents, metadatas)]

print("문서의 수:" , len(docs))
print("=" * 200)
for doc in docs[:3]:
    print(f"{doc.page_content} [출처: {doc.metadata['source']}]")
    print("-" * 200)

문서의 수: 39
[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.
----------------------------------
Rivian Automotive, Inc.는 2009년에 설립된 미국의 전기 자동차 제조업체, 자동차 기술 및 야외 레크리에이션 회사입니다.

**주요 정보:** [출처: data\Rivian_KR.md]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.
----------------------------------
- **회사 유형:** 상장
- **거래소:** NASDAQ: RIVN
- **설립:** 2009년 6월, 플로리다 주 록ledge
- **설립자:** R. J. 스캐린지
- **본사:** 미국 캘리포니아 주 어바인
- **서비스 지역:** 북미
- **주요 인물:** R. J. 스캐린지 (CEO)
- **제품:** 전기 자동차, 배터리
- **생산량 (2023):** 57,232대
- **서비스:** 전기 자동차 충전, 자동차 보험
- **수익 (2023):** 44억 3천만 미국 달러
- **순이익 (2023):** -54억 미국 달러
- **총 자산 (2023):** 168억 미국 달러 [출처: data\Rivian_KR.md]
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [9]:
from langchain_core.documents import Document  # Document 클래스 임포트

# 문자열 리스트를 Document 객체로 변환
if isinstance(korean_docs[0], str):  # 첫 번째 항목이 문자열인지 확인
    documents = [
        Document(
            page_content=json.loads(data)['page_content'],  # 문자열을 파이썬 객체로 변환
            metadata=json.loads(data)['metadata']
        ) 
        for i, data in enumerate(korean_docs)
    ]
else:
    documents = korean_docs

print(f"변환된 문서: {len(documents)}개")
pprint(documents[0])

변환된 문서: 39개
Document(metadata={'source': 'data/테슬라_KR.md', 'company': '테슬라', 'language': 'ko'}, page_content="<Document>\nTesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 이 회사는 전기 자동차(BEV), 고정형 배터리 에너지 저장 장치, 태양 전지판, 태양광 지붕널 및 관련 제품/서비스를 설계, 제조 및 판매합니다. 2003년 7월 Martin Eberhard와 Marc Tarpenning이 Tesla Motors로 설립했으며, Nikola Tesla를 기리기 위해 명명되었습니다. Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도하여 2008년에 회장 겸 CEO가 되었습니다.\n</Document>\n<Source>이 문서는 미국 전기차 회사인 '테슬라'에 대한 문서입니다.</Source>")


In [ ]:
# BM25 검색기를 사용하기 위한 준비
from langchain_community.retrievers import BM25Retriever
from ranx_k.tokenizers import KiwiTokenizer

kiwi_tokenizer = KiwiTokenizer(
    use_stopwords=False,  # 불용어 사용 안 함
    pos_filter=[]         # 품사 필터 없음
)

# Kiwi 토크나이저를 사용하는 전처리 함수
def kiwi_preprocess(text: str) -> list[str]:
    return kiwi_tokenizer.tokenize(text)  # 문자열 리스트 반환

# BM25 검색기 생성 (한국어 토크나이저 적용)
bm25_db = BM25Retriever.from_documents(
    documents,
    preprocess_func=kiwi_preprocess,
    k=4,
)


In [10]:
# BM25 검색기를 사용하기 위한 준비
from langchain_community.retrievers import BM25Retriever
from ranx_k.tokenizers import KiwiTokenizer

kiwi_tokenizer = KiwiTokenizer(
    use_stopwords=False,  # 불용어 사용 안 함
    pos_filter=[]         # 품사 필터 없음
)

# Kiwi 토크나이저를 사용하는 전처리 함수
def kiwi_preprocess(text: str) -> list[str]:
    return kiwi_tokenizer.tokenize(text)  # 문자열 리스트 반환

# BM25 검색기 생성 (한국어 토크나이저 적용)
bm25_db = BM25Retriever.from_documents(
    docs,
    preprocess_func=kiwi_preprocess,
    k=4,
)

C:\Users\JSPark\AppData\Local\Temp\ipykernel_34680\3956242132.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


In [11]:
# BM25 검색기를 사용하여 문서 검색
query = "Elon Musk는 Tesla의 초기 자금 조달과 경영 변화에 어떻게 관여했으며, 그 과정에서 어떤 논란에 직면했나요?"
retrieved_docs = bm25_db.invoke(query)

# 검색 결과 출력 
for i, doc in enumerate(retrieved_docs, 1):
    print(f"[검색 결과 {i}]")    
    print(f"{doc.page_content}\n[출처: {doc.metadata['source']}]")
    print("-"*100)

[검색 결과 1]
[출처] 이 문서는 테슬라(Tesla)에 대한 문서입니다.
----------------------------------
Tesla는 내부 고발자 보복, 근로자 권리 침해, 안전 결함, 홍보 부족, Musk의 논란의 여지가 있는 발언과 관련된 소송, 정부 조사 및 비판에 직면했습니다.

## 역사

### 창립 (2003–2004)

Tesla Motors, Inc.는 2003년 7월 1일에 Martin Eberhard와 Marc Tarpenning에 의해 설립되었으며, 각각 CEO와 CFO를 역임했습니다. Ian Wright는 얼마 지나지 않아 합류했습니다. 2004년 2월, Elon Musk는 750만 달러의 시리즈 A 자금 조달을 주도하여 회장 겸 최대 주주가 되었습니다. J. B. Straubel은 2004년 5월 CTO로 합류했습니다. 다섯 명 모두 공동 설립자로 인정받고 있습니다.

### Roadster (2005–2009)
[출처: data\Tesla_KR.md]
----------------------------------------------------------------------------------------------------
[검색 결과 2]
[출처] 이 문서는 테슬라(Tesla)에 대한 문서입니다.
----------------------------------
Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 이 회사는 전기 자동차(BEV), 고정형 배터리 에너지 저장 장치, 태양 전지판, 태양광 지붕널 및 관련 제품/서비스를 설계, 제조 및 판매합니다. 2003년 7월 Martin Eberhard와 Marc Tarpenning이 Tesla Motors로 설립했으며, Nikola Tesla를 기리기 위해 명명되었습니다. Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도하여 2008년에 회장 겸 CEO가 되었습니다.
[출처: data\Tesla_KR.md]
---------

### 3) **Emsemble Hybrid Search** 준비

- **BM25**, **벡터 검색** 결과를 **rank-fusion** 알고리즘으로 통합 (**EnsembleRetriever**)

- 각 검색기의 **순위 점수**를 고려한 최종 순위 결정

- **중복 문서** 제거와 **재순위화** 자동 수행

- 두 검색 방식의 **장점을 결합**해 검색 품질 향상

In [12]:
from langchain_classic.retrievers import EnsembleRetriever

# 검색기 초기화 
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_db, chroma_k],
    weights=[0.5, 0.5],
)


In [13]:
query = "Elon Musk는 Tesla의 초기 자금 조달과 경영 변화에 어떻게 관여했으며, 그 과정에서 어떤 논란에 직면했나요?"
retrieved_docs = hybrid_retriever.invoke(query)

# 검색 결과 출력
for doc in retrieved_docs:
    print(f"\n{doc.page_content}\n[출처: {doc.metadata['source']}]")
    print("-"*200)


[출처] 이 문서는 테슬라(Tesla)에 대한 문서입니다.
----------------------------------
Tesla는 내부 고발자 보복, 근로자 권리 침해, 안전 결함, 홍보 부족, Musk의 논란의 여지가 있는 발언과 관련된 소송, 정부 조사 및 비판에 직면했습니다.

## 역사

### 창립 (2003–2004)

Tesla Motors, Inc.는 2003년 7월 1일에 Martin Eberhard와 Marc Tarpenning에 의해 설립되었으며, 각각 CEO와 CFO를 역임했습니다. Ian Wright는 얼마 지나지 않아 합류했습니다. 2004년 2월, Elon Musk는 750만 달러의 시리즈 A 자금 조달을 주도하여 회장 겸 최대 주주가 되었습니다. J. B. Straubel은 2004년 5월 CTO로 합류했습니다. 다섯 명 모두 공동 설립자로 인정받고 있습니다.

### Roadster (2005–2009)
[출처: data\Tesla_KR.md]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

[출처] 이 문서는 테슬라(Tesla)에 대한 문서입니다.
----------------------------------
## 파트너

Tesla는 Panasonic과 파트너십을 맺고 있으며 리튬 공급에 대한 장기 계약을 맺고 있습니다. 이전 파트너로는 Daimler와 Toyota가 있습니다.

## 소송 및 논란

Tesla는 성희롱, 노동 분쟁, 사기 혐의, 대리점 분쟁, 지적 재산권, 환경 위반, 재산 피해, 인종 차별, COVID-19 팬데믹 대응 및 수리 권리와 관련된 소송 및 논란에 직면했습니다.

## 비판

Tesl

### 4) **RAG 체인**

- **답변**과 **검색 문서**를 함께 출력

In [14]:
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.runnables import RunnableConfig, RunnablePassthrough, RunnableParallel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from typing import List, Dict

def rag_bot(
    question: str,
    retriever: BaseRetriever,
    llm: BaseChatModel,
    config: RunnableConfig | None = None,
) -> Dict[str, str | List[Document]]:
    """
    문서 검색 기반 질의응답 수행
    """
    docs = retriever.invoke(question)
    context = "\n".join(doc.page_content for doc in docs)

    system_prompt = f"""문서 기반 질의응답 어시스턴트입니다.
- 제공된 문서만 참고하여 답변
- 불확실할 경우 '모르겠습니다' 라고 응답
- 3문장 이내로 답변
{context}"""

    prompt = ChatPromptTemplate.from_messages(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": "###문서\n{question}\n\n###질문\n{question}\n"},
        ]
    )

    docqa_chain = {
        "context": lambda x: context,
        "question": RunnablePassthrough(),
        "docs": lambda x: docs,
    } | RunnableParallel({
        "answer": prompt | llm | StrOutputParser(),
        "documents": lambda x: x["docs"],
    })

    return docqa_chain.invoke(question, config=config)

In [15]:
from langchain_openai import ChatOpenAI

# 모델 생성
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# RAG 체인 실행
rag_bot(
    question="Elon Musk는 Tesla의 초기 자금 조달과 경영 변화에 어떻게 관여했으며, 그 과정에서 어떤 논란에 직면했나요?",
    retriever=hybrid_retriever,
    llm=llm,
    config={"callbacks": [langfuse_handler]},   # 콜백 핸들러 추가
)

{'answer': 'Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도하여 회장 겸 최대 주주가 되었고, 2008년에는 CEO로 취임했습니다. 그는 경영 변화 과정에서 2007년과 2008년에 기존 CEO들이 물러나도록 했으며, 2009년에는 공동 설립자 Martin Eberhard가 Musk를 상대로 소송을 제기했으나 기각되었습니다. 이 과정에서 Musk는 경영권 인수와 관련된 논란에 직면했습니다.',
 'documents': [Document(metadata={'language': 'ko', 'company': '테슬라(Tesla)', 'source': 'data\\Tesla_KR.md'}, page_content='[출처] 이 문서는 테슬라(Tesla)에 대한 문서입니다.\n----------------------------------\nTesla는 내부 고발자 보복, 근로자 권리 침해, 안전 결함, 홍보 부족, Musk의 논란의 여지가 있는 발언과 관련된 소송, 정부 조사 및 비판에 직면했습니다.\n\n## 역사\n\n### 창립 (2003–2004)\n\nTesla Motors, Inc.는 2003년 7월 1일에 Martin Eberhard와 Marc Tarpenning에 의해 설립되었으며, 각각 CEO와 CFO를 역임했습니다. Ian Wright는 얼마 지나지 않아 합류했습니다. 2004년 2월, Elon Musk는 750만 달러의 시리즈 A 자금 조달을 주도하여 회장 겸 최대 주주가 되었습니다. J. B. Straubel은 2004년 5월 CTO로 합류했습니다. 다섯 명 모두 공동 설립자로 인정받고 있습니다.\n\n### Roadster (2005–2009)'),
  Document(metadata={'source': 'data\\Tesla_KR.md', 'company': '테슬라(Tesla)', 'language': 'ko'}, page_content='[출처] 이 문서는 테슬라(Tesla)에 대한 문서입니다.\n--------

---
### **[실습]**

- gemin-2.5-flash-lite 모델과 벡터스토어 검색기를 사용하여 RAG 체인을 실행합니다.
- 실행 결과를 langfuse UI에서 확인하고, gpt-4.1-mini 모델의 답변과 비교합니다.

In [16]:
# 여기에 코드를 작성하세요.

from langchain_google_genai import ChatGoogleGenerativeAI

# 모델 생성
gemin_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)

# RAG 체인 실행
rag_bot(
    question="Elon Musk는 Tesla의 초기 자금 조달과 경영 변화에 어떻게 관여했으며, 그 과정에서 어떤 논란에 직면했나요?",
    retriever=hybrid_retriever,
    llm=gemin_llm,
    config={"callbacks": [langfuse_handler]},   # 콜백 핸들러 추가
)

{'answer': 'Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도하여 회장 겸 최대 주주가 되었습니다. 2008년 10월에는 CEO가 되었으며, 이 과정에서 공동 설립자인 Martin Eberhard는 Musk를 상대로 소송을 제기했지만 나중에 기각되었습니다.',
 'documents': [Document(metadata={'language': 'ko', 'company': '테슬라(Tesla)', 'source': 'data\\Tesla_KR.md'}, page_content='[출처] 이 문서는 테슬라(Tesla)에 대한 문서입니다.\n----------------------------------\nTesla는 내부 고발자 보복, 근로자 권리 침해, 안전 결함, 홍보 부족, Musk의 논란의 여지가 있는 발언과 관련된 소송, 정부 조사 및 비판에 직면했습니다.\n\n## 역사\n\n### 창립 (2003–2004)\n\nTesla Motors, Inc.는 2003년 7월 1일에 Martin Eberhard와 Marc Tarpenning에 의해 설립되었으며, 각각 CEO와 CFO를 역임했습니다. Ian Wright는 얼마 지나지 않아 합류했습니다. 2004년 2월, Elon Musk는 750만 달러의 시리즈 A 자금 조달을 주도하여 회장 겸 최대 주주가 되었습니다. J. B. Straubel은 2004년 5월 CTO로 합류했습니다. 다섯 명 모두 공동 설립자로 인정받고 있습니다.\n\n### Roadster (2005–2009)'),
  Document(metadata={'source': 'data\\Tesla_KR.md', 'company': '테슬라(Tesla)', 'language': 'ko'}, page_content='[출처] 이 문서는 테슬라(Tesla)에 대한 문서입니다.\n----------------------------------\n## 파트너\n\nTesla는 Panasonic과 파트너십을 맺고 있으며 리튬 공급에 대

---

## **Comparison (비교 평가)**

- **OpenEvals** 비교 평가기로 동일 입력에 대한 여러 모델/프롬프트의 출력을 객관적으로 비교
- **A/B 테스트**와 **선호도 점수** 생성에 활용
- 참조: [langchain-ai/openevals](https://github.com/langchain-ai/openevals)


### **1) Reference-free**

- **평가 특징**: 참조 답변 없이 두 RAG 답변 직접 비교
- **평가 요소**: 사실성, 관련성, 일관성 등 상대 비교
- **장점**: 절대 기준 없이도 RAG 시스템 간 성능 차이 판단 가능 (객관적 비교 가능)

`(1) A/B 테스트 평가 - 기본 개요`

In [17]:
from openevals.llm import create_llm_as_judge

# Pairwise 비교를 위한 프롬프트 정의
# 핵심: choices=[0.0, 0.5, 1.0]으로 A/B/TIE를 숫자에 매핑
PAIRWISE_COMPARISON_PROMPT = """You are comparing two AI assistant responses to a question.

<Question>
{input}
</Question>

<Response A>
{prediction}
</Response A>

<Response B>
{prediction_b}
</Response B>

Compare the two responses based on:
1. Accuracy and correctness
2. Helpfulness and relevance
3. Clarity and conciseness

<Rubric>
Assign a score based on which response is better:
- 0: Response A is clearly better
- 0.5: Both responses are equally good (TIE)
- 1: Response B is clearly better
</Rubric>
"""

# 결과 해석 헬퍼 함수
def interpret_pairwise(score):
    """Pairwise 점수를 사람이 읽을 수 있는 라벨로 변환"""
    if score == 0.0:
        return "A 승 (Response A is better)"
    elif score == 1.0:
        return "B 승 (Response B is better)"
    else:
        return "무승부 (TIE)"

# 비교 평가기 생성 (choices로 3단계 척도 지정)
pairwise_evaluator = create_llm_as_judge(
    prompt=PAIRWISE_COMPARISON_PROMPT,
    feedback_key="pairwise_comparison",
    choices=[0.0, 0.5, 1.0],   # 0=A 승, 0.5=TIE, 1=B 승
    model="openai:gpt-4.1-mini",
)

# 두 모델의 출력 비교
result = pairwise_evaluator(
    input="파이썬이 무엇인지 설명해주세요.",
    prediction="파이썬은 읽기 쉽고 간단한 문법을 가진 프로그래밍 언어입니다.",
    prediction_b="파이썬은 동적 타이핑을 지원하는 고수준 프로그래밍 언어로, 데이터 과학과 웹 개발에 널리 사용됩니다.",
)

print(f"평가 키: {result['key']}")
print(f"점수: {result['score']} → {interpret_pairwise(result['score'])}")
print("-"*100)
print(f"평가 근거: {result['comment']}")

평가 키: pairwise_comparison
점수: 1.0 → B 승 (Response B is better)
----------------------------------------------------------------------------------------------------
평가 근거: Response A provides a simple explanation that Python is a programming language with readable and simple syntax. This is accurate and clear but somewhat limited in detail. Response B gives a more detailed description, mentioning Python's dynamic typing, being a high-level programming language, and its common uses in data science and web development. This information is also accurate and adds helpful context relevant to understanding what Python is and where it is used. Both responses are clear and concise, but Response B offers greater helpfulness and relevance with additional correct details, making it a better answer overall. Thus, the score should be: 1.0.


`(2) A/B 테스트 평가 - 모델 비교`

In [18]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

# LLM 모델 생성
gpt4mini_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
gemini2flash_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)

# RAG 체인 실행
question = "Elon Musk는 Tesla의 초기 자금 조달과 경영 변화에 어떻게 관여했으며, 그 과정에서 어떤 논란에 직면했나요?"
gpt_response = rag_bot(
    question=question,
    retriever=hybrid_retriever,
    llm=gpt4mini_llm,
    config={
        "callbacks": [langfuse_handler],
        "tags": ["rag_bot", "evaluation"],
        "metadata": {
            "model": "gpt-4.1-mini",
            "temperature": 0,
            },
        },
)


gemini_response = rag_bot(
    question=question,
    retriever=hybrid_retriever,
    llm=gemini2flash_llm,
    config={
        "callbacks": [langfuse_handler],
        "tags": ["rag_bot", "evaluation"],
        "metadata": {
            "model": "gemini-2.5-flash-lite",
            "temperature": 0,
            },
        },
)

# 비교 평가기 생성 : OpenAI gpt-4.1-mini vs Google Gemini-2.5-flash
pairwise_evaluator = create_llm_as_judge(
    prompt=PAIRWISE_COMPARISON_PROMPT,
    feedback_key="pairwise_comparison",
    choices=[0.0, 0.5, 1.0],
    model="openai:gpt-4.1-mini",
)

# 두 모델의 출력 비교
result = pairwise_evaluator(
    input=question,
    prediction=gpt_response["answer"],
    prediction_b=gemini_response["answer"],
)

print(f"[A] gpt-4.1-mini: {gpt_response['answer']}")
print(f"[B] Gemini-2.5-flash: {gemini_response['answer']}")
print("-"*100)
print(f"결과: {result['score']} → {interpret_pairwise(result['score'])}")
print(f"근거: {result['comment']}")

[A] gpt-4.1-mini: Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도하여 회장 겸 최대 주주가 되었고, 2008년에는 CEO로 취임했습니다. 그는 경영 변화 과정에서 2007년과 2008년에 기존 CEO들이 물러나도록 했으며, 2009년에는 공동 설립자 Martin Eberhard가 Musk를 상대로 소송을 제기했으나 기각되었습니다. 이 과정에서 Musk는 경영권 인수와 관련된 논란에 직면했습니다.
[B] Gemini-2.5-flash: Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도하여 회장 겸 최대 주주가 되었습니다. 2008년 10월에는 CEO가 되었으며, 이 과정에서 공동 설립자인 Martin Eberhard는 Musk를 상대로 소송을 제기했지만 나중에 기각되었습니다.
----------------------------------------------------------------------------------------------------
결과: 0 → A 승 (Response A is better)
근거: Response A provides a more comprehensive overview of Elon Musk's involvement with Tesla by including detailed information about the years (2007 and 2008) when previous CEOs stepped down, the fact that Musk became CEO in 2008, and elaborates on the controversy involving Martin Eberhard suing Musk, indicating that the lawsuit was related to management changes. Response A also explicitly mentions Musk's role as chairman and largest shareholder, align

---
### **[실습]**

- ollama와 groq에서 각각 1개의 모델을 선택하고, RAG 체인을 구성합니다.
- 두 모델의 실행 결과를 비교합니다. (langfuse UI 확인)

In [36]:
# 여기에 코드를 작성하세요.
from langchain_ollama import ChatOllama
from langchain_groq import ChatGroq
from openevals.llm import create_llm_as_judge

# 1. 모델 생성 (로컬 Ollama 및 Groq 클라우드 모델)
ollama_llm = ChatOllama(
    model="exaone3.5:7.8b",  # 로컬에 설치 및 pull 완료된 Ollama 모델명 입력
    temperature=0
)

groq_llm = ChatGroq(
    model="llama-3.3-70b-versatile",  # 사용하고자 하는 Groq 모델명 입력
    temperature=0
)

# 2. 질문 정의 및 RAG 체인 실행
question = "Elon Musk는 Tesla의 초기 자금 조달과 경영 변화에 어떻게 관여했으며, 그 과정에서 어떤 논란에 직면했나요?"

# Ollama 모델 실행
ollama_response = rag_bot(
    question=question,
    retriever=hybrid_retriever,
    llm=ollama_llm,
    config={
        "callbacks": [langfuse_handler],
        "tags": ["rag_bot", "evaluation", "ollama"],
        "metadata": {
            "model": "exaone3.5:7.8b",
            "temperature": 0,
        },
    },
)

# Groq 모델 실행
groq_response = rag_bot(
    question=question,
    retriever=hybrid_retriever,
    llm=groq_llm,
    config={
        "callbacks": [langfuse_handler],
        "tags": ["rag_bot", "evaluation", "groq"],
        "metadata": {
            "model": "llama-3.3-70b-versatile",
            "temperature": 0,
        },
    },
)

# 3. GPT 기반 비교 평가기 생성 (기존 PAIRWISE_COMPARISON_PROMPT 활용)
pairwise_evaluator = create_llm_as_judge(
    prompt=PAIRWISE_COMPARISON_PROMPT,
    feedback_key="pairwise_comparison_ollama_vs_groq",
    choices=[0.0, 0.5, 1.0],
    model="openai:gpt-4.1-mini",
)

# 4. 두 모델의 출력 비교 (A: Ollama, B: Groq)
result = pairwise_evaluator(
    input=question,
    prediction=ollama_response["answer"],
    prediction_b=groq_response["answer"],
)

# 5. 결과 출력
print(f"[A] Ollama (exaone3.5:7.8b): {ollama_response['answer']}")
print(f"[B] Groq (llama-3.3-70b-versatile): {groq_response['answer']}")
print("-" * 100)
print(f"결과: {result['score']} → {interpret_pairwise(result['score'])}")
print(f"근거: {result['comment']}")


[A] Ollama (exaone3.5:7.8b): Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도하며 시리즈 A 자금 조달에서 750만 달러를 이끌어냈고, 이후 2008년에 회장 겸 CEO로 취임하여 회사의 전략적 방향과 경영 전반에 적극적으로 관여했습니다. 그의 리더십 아래 Tesla는 전기 자동차 개발에 집중하고, 기술 혁신과 대중 시장 진출을 추진했습니다. 그러나 이러한 과정에서 다음과 같은 논란에 직면했습니다:

1. **경영 변화와 내부 갈등**: 초기 공동 설립자인 Martin Eberhard와 Marc Tarpenning이 경영 변화 과정에서 퇴사하면서 내부 갈등이 발생했습니다. Eberhard는 Musk를 상대로 소송을 제기했으나 기각되었습니다.
2. **경영 스타일 논란**: Musk의 강한 리더십 스타일과 때때로 논란의 여지가 있는 발언들로 인해 직원들과의 갈등, 그리고 외부 비판을 받았습니다.
3. **기술 및 안전 문제**: Autopilot 기능과 관련된 충돌 사고 및 소프트웨어 해킹 논란 등 기술적 문제와 안전 결함에 대한 비판이 있었습니다.
4. **재무 및 투자 논란**: Tesla의 주식 가격 변동과 관련된 공매도자들의 비판, 그리고 회사의 재무 상태에 대한 의문 제기 등이 있었습니다.
[B] Groq (llama-3.3-70b-versatile): Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도하여 회장 겸 최대 주주가 되었습니다. 그는 2008년에 CEO가 되었습니다. 그는 Tesla의 경영 변화에 관여했으며, 그 과정에서 여러 논란에 직면했습니다. 예를 들어, 그는 Martin Eberhard와의 분쟁으로 인해 소송을 제기받았습니다. 또한 그는 Tesla의 성장과 발전에 중요한 역할을 했습니다.
----------------------------------------------------------------------------------------------------
결과: 0.0 

`(3) A/B 테스트 평가 - 프롬프트 비교`

In [20]:
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.runnables import RunnableConfig
from typing import List, Dict

def rag_bot_b(
    question: str,
    retriever: BaseRetriever,
    llm: BaseChatModel,
    config: RunnableConfig | None = None,
) -> Dict[str, str | List[Document]]:
    """
    문서 검색 기반 질의응답 수행
    """
    docs = retriever.invoke(question)
    context = "\n".join(doc.page_content for doc in docs)

    system_prompt = f"""RAG Assistant to answer questions based on provided documents.

    Guidelines:
    - Reference only provided context
    - Reply "I don't know" if uncertain
    - Keep responses under 3 sentences
    - Answer in 한국어

    Context:
    {context}"""

    prompt = ChatPromptTemplate.from_messages(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": "\n\n[Question]{question}\n\n[Answer]\n"},
        ]
    )

    docqa_chain = {
        "context": lambda x: context,
        "question": RunnablePassthrough(),
        "docs": lambda x: docs,
    } | RunnableParallel({
        "answer": prompt | llm | StrOutputParser(),
        "documents": lambda x: x["docs"],
    })

    return docqa_chain.invoke(question, config=config)

In [21]:
# RAG 체인 실행
question = "Elon Musk는 Tesla의 초기 자금 조달과 경영 변화에 어떻게 관여했으며, 그 과정에서 어떤 논란에 직면했나요?"
gpt_prompt_b_response = rag_bot_b(
    question=question,
    retriever=hybrid_retriever,
    llm=gpt4mini_llm,
    config={
        "callbacks": [langfuse_handler],
        "tags": ["rag_bot", "evaluation", "prompt_b"],
        "langfuse_metadata": {
            "model": "gpt-4.1-mini",
            "temperature": 0,
        },
    },
)

# 비교 평가기 생성 (프롬프트 성능 비교)
pairwise_evaluator = create_llm_as_judge(
    prompt=PAIRWISE_COMPARISON_PROMPT,
    feedback_key="pairwise_comparison",
    choices=[0.0, 0.5, 1.0],
    model="openai:gpt-4.1-mini",
)

# 두 모델의 출력 비교 (프롬프트 성능 비교)
result = pairwise_evaluator(
    input=question,
    prediction=gpt_response["answer"],
    prediction_b=gpt_prompt_b_response["answer"],
)

print(f"[A] 한국어 프롬프트: {gpt_response['answer']}")
print(f"[B] 영어 프롬프트: {gpt_prompt_b_response['answer']}")
print("-"*100)
print(f"결과: {result['score']} → {interpret_pairwise(result['score'])}")
print(f"근거: {result['comment']}")

[A] 한국어 프롬프트: Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도하여 회장 겸 최대 주주가 되었고, 2008년에는 CEO로 취임했습니다. 그는 경영 변화 과정에서 2007년과 2008년에 기존 CEO들이 물러나도록 했으며, 2009년에는 공동 설립자 Martin Eberhard가 Musk를 상대로 소송을 제기했으나 기각되었습니다. 이 과정에서 Musk는 경영권 인수와 관련된 논란에 직면했습니다.
[B] 영어 프롬프트: Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도하여 회장 겸 최대 주주가 되었고, 2008년 10월 CEO로 인수했습니다. 그는 Roadster 개발 전략에 적극 참여했으며, 2009년 공동 설립자 Martin Eberhard가 Musk를 상대로 소송을 제기했으나 기각되었습니다. 이 과정에서 경영권 변화와 관련된 논란에 직면했습니다.
----------------------------------------------------------------------------------------------------
결과: 0.5 → 무승부 (TIE)
근거: Both Response A and Response B accurately describe Elon Musk's involvement in Tesla's initial funding and management changes, including the 2009 lawsuit by Martin Eberhard and its dismissal. Response A provides a slightly clearer timeline (2007-2008 CEO changes, 2008 CEO appointment) and explicitly mentions Musk becoming CEO in 2008, whereas Response B says "2008년 10월 CEO로 인수했습니다," which is less precise phrasing. Response B

---
### **[실습]**

- 두 가지 버전의 프롬프트를 작성하고, 각각 별도의 RAG 체인을 구성합니다. (모델은 공통 적용)
- 두 가지 실행 결과를 비교합니다. (langfuse UI 확인)

In [ ]:
# 여기에 코드를 작성하세요.



### **2) Reference-based**

- **기준 활용**: 참조 답안과 RAG 응답을 비교 평가
- **평가 방식**: 자동화된 A/B 테스트로 객관적 성능 측정
- **주요 지표**: 정확도, 완성도, 관련성 등 정량적 평가
- 참조 답안 기반 **체계적인 품질 평가** 수행

`(1) A/B 테스트 평가 - 모델 비교`

In [22]:
# Reference-based Pairwise 비교를 위한 프롬프트 정의
PAIRWISE_COMPARISON_WITH_REFERENCE_PROMPT = """You are comparing two AI assistant responses to a question with a reference answer.

<Question>
{input}
</Question>

<Reference Answer>
{reference}
</Reference Answer>

<Response A>
{prediction}
</Response A>

<Response B>
{prediction_b}
</Response B>

Compare the two responses against the reference answer based on:
1. Accuracy compared to the reference
2. Completeness of information
3. Relevance to the question

<Rubric>
Assign a score based on which response is better:
- 0: Response A is clearly better (more accurate to reference)
- 0.5: Both responses are equally good (TIE)
- 1: Response B is clearly better (more accurate to reference)
</Rubric>
"""

# 참조 답안이 있는 평가기 생성
evaluator = create_llm_as_judge(
    prompt=PAIRWISE_COMPARISON_WITH_REFERENCE_PROMPT,
    feedback_key="pairwise_with_reference",
    choices=[0.0, 0.5, 1.0],
    model="openai:gpt-4.1-mini",
)

# 참조 답안
ground_truth = """Elon Musk는 2004년 2월에 750만 달러의 시리즈 A 자금 조달을 주도하여 Tesla의 회장 겸 최대 주주가 되었습니다.
그는 주류 차량으로 확장하기 전에 프리미엄 스포츠카로 시작하는 전략에 초점을 맞춰 적극적인 역할을 수행했습니다. 2008년 10월에는 CEO로 인수했습니다.
그러나 Tesla는 내부 고발자 보복, 근로자 권리 침해, 안전 결함, 홍보 부족, Musk의 논란의 여지가 있는 발언과 관련된 소송, 정부 조사 및 비판에 직면했습니다."""

# 두 모델의 출력 비교
result = evaluator(
    input=question,
    prediction=gpt_response["answer"],        # gpt-4.1-mini 응답 (A)
    prediction_b=gemini_response["answer"],   # Gemini-2.5-flash 응답 (B)
    reference=ground_truth  # 참조 답안 (정답)
)

print(f"[A] gpt-4.1-mini: {gpt_response['answer']}")
print(f"[B] Gemini-2.5-flash: {gemini_response['answer']}")
print("-"*100)
print(f"결과: {result['score']} → {interpret_pairwise(result['score'])}")
print(f"근거: {result['comment']}")

[A] gpt-4.1-mini: Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도하여 회장 겸 최대 주주가 되었고, 2008년에는 CEO로 취임했습니다. 그는 경영 변화 과정에서 2007년과 2008년에 기존 CEO들이 물러나도록 했으며, 2009년에는 공동 설립자 Martin Eberhard가 Musk를 상대로 소송을 제기했으나 기각되었습니다. 이 과정에서 Musk는 경영권 인수와 관련된 논란에 직면했습니다.
[B] Gemini-2.5-flash: Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도하여 회장 겸 최대 주주가 되었습니다. 2008년 10월에는 CEO가 되었으며, 이 과정에서 공동 설립자인 Martin Eberhard는 Musk를 상대로 소송을 제기했지만 나중에 기각되었습니다.
----------------------------------------------------------------------------------------------------
결과: 0 → A 승 (Response A is better)
근거: Response A provides more detailed and complete information compared to Response B. It mentions the year when Elon Musk led the initial fundraising (2004) and became chairman and largest shareholder, consistent with the reference answer. It also details Musk's management changes, specifically the departure of CEOs in 2007 and 2008, and the 2009 lawsuit from Martin Eberhard, which aligns with the reference's mention of litigation and controversies. 

### **[실습]**

- 테스트셋(df_qa_test)에서 하나의 샘플을 선택합니다.
- 이 샘플에 대한 RAG 답변을 두 가지 모델로부터 구합니다.
- 두 가지 실행결과에 대한 A/B 테스트 분석을 수행합니다. (langfuse UI 확인)

In [32]:
df_qa_test.head(3)

,user_input,reference_contexts,reference,synthesizer_name
0,"Tesla, Inc.는 미국에서 어떤 역할을 하고 있으며, 이 회사의 주요 제품과 ...","['Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 이 회...","Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사로, 전기 자동차(...",single_hop_specifc_query_synthesizer
1,Forbes Global 2000에서 테슬라 순위 뭐야?,['Tesla의 차량 생산은 2008년 Roadster로 시작하여 Model S (...,테슬라는 Forbes Global 2000에서 69위에 랭크되었습니다.,single_hop_specifc_query_synthesizer
2,Tesla는 언제 누가 만들었나?,"['Tesla는 내부 고발자 보복, 근로자 권리 침해, 안전 결함, 홍보 부족, M...","Tesla Motors, Inc.는 2003년 7월 1일에 Martin Eberha...",single_hop_specifc_query_synthesizer


In [33]:
# 여기에 코드를 작성하세요.

# 1. 테스트셋(df_qa_test)에서 하나의 샘플 선택
# (df_qa_test의 컬럼명에 맞춰 'question', 'answer'를 지정합니다)

sample = df_qa_test.iloc[0]  # 첫 번째 샘플 선택
sample_question = sample["user_input"]
sample_reference = sample["reference"]  # 참조 답안(Ground Truth)
print(f"선택된 질문: {sample_question}")
print(f"참조 답변: {sample_reference}")
print("=" * 100)


선택된 질문: Tesla, Inc.는 미국에서 어떤 역할을 하고 있으며, 이 회사의 주요 제품과 서비스는 무엇인가요?
참조 답변: Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사로, 전기 자동차(BEV), 고정형 배터리 에너지 저장 장치, 태양 전지판, 태양광 지붕널 및 관련 제품/서비스를 설계, 제조 및 판매합니다.


In [34]:

# 2. 두 모델(gpt-4.1-mini & gemini-2.5-flash-lite)로부터 RAG 답변 구하기
# (이전 실습에서 생성한 ollama_llm과 groq_llm을 사용하고 싶다면 해당 변수로 대체하셔도 됩니다)
response_a = rag_bot(
    question=sample_question,
    retriever=hybrid_retriever,
    llm=gpt4mini_llm,
    config={
        "callbacks": [langfuse_handler],
        "tags": ["rag_bot", "evaluation", "gpt-4.1-mini"],
        "metadata": {
            "model": "gpt-4.1-mini",
            "temperature": 0,
        },
    },
)
response_b = rag_bot(
    question=sample_question,
    retriever=hybrid_retriever,
    llm=gemini2flash_llm,
    config={
        "callbacks": [langfuse_handler],
        "tags": ["rag_bot", "evaluation", "gemini-2.5-flash-lite"],
        "metadata": {
            "model": "gemini-2.5-flash-lite",
            "temperature": 0,
        },
    },
)

In [35]:
# 3. Reference-based Pairwise 평가기(evaluator)를 통한 A/B 테스트 분석
result = evaluator(
    input=sample_question,
    prediction=response_a["answer"],
    prediction_b=response_b["answer"],
    reference=sample_reference
)
# 4. 결과 출력
print(f"[A] GPT-4.1-mini: {response_a['answer']}")
print(f"[B] Gemini-2.5-flash-lite: {response_b['answer']}")
print("-" * 100)
print(f"결과: {result['score']} → {interpret_pairwise(result['score'])}")
print(f"근거: {result['comment']}")

[A] GPT-4.1-mini: Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사로, 전기 자동차, 고정형 배터리 에너지 저장 장치, 태양 전지판, 태양광 지붕널 등을 설계, 제조 및 판매합니다. 주요 서비스로는 전기차 충전 네트워크인 Supercharger와 Destination 충전 위치, 차량 보험 서비스, 차량 원격 진단 및 수리, 에너지 제품 설치 및 관리 등이 있습니다. 또한, 태양광 및 배터리 에너지 저장 시스템을 포함한 청정 에너지 솔루션도 제공합니다.
[B] Gemini-2.5-flash-lite: Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 이 회사는 전기 자동차, 고정형 배터리 에너지 저장 장치, 태양 전지판, 태양광 지붕널 및 관련 제품/서비스를 설계, 제조 및 판매합니다. 또한 Tesla는 Supercharger 네트워크, Destination 충전 위치 네트워크, 차량 서비스, 보험 서비스 및 에너지 제품을 제공합니다.
----------------------------------------------------------------------------------------------------
결과: 0 → A 승 (Response A is better)
근거: Both Response A and Response B accurately describe Tesla, Inc. as a US multinational automobile and clean energy company, and both list the main products including electric vehicles, stationary battery energy storage, solar panels, and solar roof tiles, which aligns well with the reference answer. However, Response A provides additional detailed informat

`(2) 사용자 정의 기준으로 A/B 테스트 평가 - 모델 비교`

In [37]:
# 사용자 정의 평가 기준을 포함하는 프롬프트 정의
CUSTOM_CRITERIA_PAIRWISE_PROMPT = """You are comparing two AI assistant responses to a question with a reference answer.

Evaluation Criteria:
- 간결성 (Conciseness): 문장이 간단하고 불필요한 내용이 없는가?
- 명확성 (Clarity): 문장이 명확하고 이해하기 쉬운가?
- 정확성 (Accuracy): 내용이 정확하고 사실에 부합하는가?
- 적절성 (Appropriateness): 글의 어조와 스타일이 적절한가?

<Question>
{input}
</Question>

<Reference Answer>
{reference}
</Reference Answer>

<Response A>
{prediction}
</Response A>

<Response B>
{prediction_b}
</Response B>

<Rubric>
Based on the evaluation criteria above, assign a score:
- 0: Response A is clearly better across the criteria
- 0.5: Both responses are equally good (TIE)
- 1: Response B is clearly better across the criteria
</Rubric>

Provide a detailed explanation for your choice based on each criterion."""

# 사용자 정의 평가 기준을 사용하여 평가기 생성
evaluator = create_llm_as_judge(
    prompt=CUSTOM_CRITERIA_PAIRWISE_PROMPT,
    feedback_key="custom_criteria_pairwise",
    choices=[0.0, 0.5, 1.0],
    model="openai:gpt-4.1-mini",
)

# 두 모델의 출력 비교
result = evaluator(
    input=question,
    prediction=gpt_response["answer"],        # gpt-4.1-mini 응답 (A)
    prediction_b=gemini_response["answer"],   # Gemini-2.5-flash 응답 (B)
    reference=ground_truth  # 참조 답안 (정답)
)

print(f"[A] gpt-4.1-mini: {gpt_response['answer']}")
print(f"[B] Gemini-2.5-flash: {gemini_response['answer']}")
print("-"*100)
print(f"결과: {result['score']} → {interpret_pairwise(result['score'])}")
print(f"근거: {result['comment']}")

[A] gpt-4.1-mini: Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도하여 회장 겸 최대 주주가 되었고, 2008년에는 CEO로 취임했습니다. 그는 경영 변화 과정에서 2007년과 2008년에 기존 CEO들이 물러나도록 했으며, 2009년에는 공동 설립자 Martin Eberhard가 Musk를 상대로 소송을 제기했으나 기각되었습니다. 이 과정에서 Musk는 경영권 인수와 관련된 논란에 직면했습니다.
[B] Gemini-2.5-flash: Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도하여 회장 겸 최대 주주가 되었습니다. 2008년 10월에는 CEO가 되었으며, 이 과정에서 공동 설립자인 Martin Eberhard는 Musk를 상대로 소송을 제기했지만 나중에 기각되었습니다.
----------------------------------------------------------------------------------------------------
결과: 1 → B 승 (Response B is better)
근거: - 간결성 (Conciseness):
Response B는 Response A보다 더 간결합니다. 불필요한 세부사항(예: 2007년과 2008년에 기존 CEO들이 물러나도록 했다는 내용)은 Reference Answer에 나타나지 않으며, Response A에만 포함되어 있어 다소 부수적인 정보로 보입니다. Response B는 핵심 내용에 집중하여 간결합니다.

- 명확성 (Clarity):
양쪽 모두 문장이 명확하고 이해하기 쉽습니다. 그러나 Response B는 불필요한 세부사항이 없고, 정보가 더 직관적으로 제시되어 가독성이 좋습니다.

- 정확성 (Accuracy):
양쪽 모두 Reference Answer에 부합하는 내용을 담고 있습니다. Response A는 2007년과 2008년에 CEO들이 물러난 사실을 추가했는데, 해당 내용은 Reference Answer에 없으므로 다소 과잉 

---
### **[실습]**

- 테스트셋(df_qa_test)에서 하나의 샘플을 선택합니다.
- 이 샘플에 대한 RAG 답변을 두 가지 모델로부터 구합니다.
- 두 가지 실행결과에 대한 A/B 테스트 분석을 수행합니다. (langfuse UI 확인)

In [ ]:
# 여기에 코드를 작성하세요.

`(3) 사용자 정의 프롬프트로 A/B 테스트 평가 - 모델 비교`

In [38]:
# 사용자 정의 프롬프트 (한국어, 5단계 척도)
DETAILED_CUSTOM_PAIRWISE_PROMPT = """주어진 입력 맥락에서 A와 B 중 어느 것이 더 나은지 평가하시오.

다음 기준에 따라 평가하시오:
- 간결성 (Conciseness): 문장이 간단하고 불필요한 내용이 없는가?
- 명확성 (Clarity): 문장이 명확하고 이해하기 쉬운가?
- 정확성 (Accuracy): 내용이 정확하고 사실에 부합하는가?
- 적절성 (Appropriateness): 글의 어조와 스타일이 적절한가?

데이터
----
입력: {input}
참조: {reference}
A: {prediction}
B: {prediction_b}
----

<Rubric>
단계별로 평가한 후, 점수를 부여하시오:
- 0: A가 명확히 우수
- 0.25: A가 약간 우수
- 0.5: 동등 (무승부)
- 0.75: B가 약간 우수
- 1: B가 명확히 우수
</Rubric>

각 기준에 대해 단계별로 평가하고, 최종 판단을 내리시오.
"""

# 5단계 척도 해석 함수
def interpret_5scale(score):
    labels = {0.0: "A 명확 우수", 0.25: "A 약간 우수", 0.5: "무승부", 0.75: "B 약간 우수", 1.0: "B 명확 우수"}
    return labels.get(score, f"점수: {score}")

# 5단계 평가기 생성
evaluator = create_llm_as_judge(
    prompt=DETAILED_CUSTOM_PAIRWISE_PROMPT,
    feedback_key="detailed_custom_pairwise",
    choices=[0.0, 0.25, 0.5, 0.75, 1.0],   # 5단계 척도
    model="openai:gpt-4.1-mini",
)

# 두 모델의 출력 비교
result = evaluator(
    input=question,
    prediction=gpt_response["answer"],        # gpt-4.1-mini 응답 (A)
    prediction_b=gemini_response["answer"],   # Gemini-2.5-flash 응답 (B)
    reference=ground_truth  # 참조 답안 (정답)
)

print(f"[A] gpt-4.1-mini: {gpt_response['answer']}")
print(f"[B] Gemini-2.5-flash: {gemini_response['answer']}")
print("-"*100)
print(f"결과: {result['score']} → {interpret_5scale(result['score'])}")
print(f"근거: {result['comment']}")

[A] gpt-4.1-mini: Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도하여 회장 겸 최대 주주가 되었고, 2008년에는 CEO로 취임했습니다. 그는 경영 변화 과정에서 2007년과 2008년에 기존 CEO들이 물러나도록 했으며, 2009년에는 공동 설립자 Martin Eberhard가 Musk를 상대로 소송을 제기했으나 기각되었습니다. 이 과정에서 Musk는 경영권 인수와 관련된 논란에 직면했습니다.
[B] Gemini-2.5-flash: Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도하여 회장 겸 최대 주주가 되었습니다. 2008년 10월에는 CEO가 되었으며, 이 과정에서 공동 설립자인 Martin Eberhard는 Musk를 상대로 소송을 제기했지만 나중에 기각되었습니다.
----------------------------------------------------------------------------------------------------
결과: 0.75 → B 약간 우수
근거: Conciseness: B is more concise than A because it provides the necessary information without additional details like the replacement of CEOs in 2007 and 2008 mentioned in A. A includes more detail but is longer.

Clarity: Both A and B are clear, but B's concise structure may make it easier to follow. A's additional details could cause slight complexity.

Accuracy: Both responses align with the reference data. A adds details about CEO replacements and the 2009 laws

---

## **점수 척도 Pairwise 평가**

- 이진 선택(A/B/TIE) 대신 **연속 점수**로 품질 차이를 측정
- 두 답변 각각에 점수를 부여하여 세밀한 비교 가능


In [39]:
# 점수 척도 Pairwise 평가 - continuous (0~1 연속 점수)

from openevals.llm import create_llm_as_judge

PAIRWISE_SCORE_PROMPT = """두 AI 답변을 비교하여 0~1 사이 연속 점수를 부여하세요.

<질문>
{input}
</질문>

<참조 답변>
{reference}
</참조 답변>

<답변 A>
{prediction}
</답변 A>

<답변 B>
{prediction_b}
</답변 B>

<평가 기준>
- 정확성: 참조 답변과의 일치도
- 완전성: 필요한 정보 포함 여부
- 간결성: 불필요한 내용 없이 핵심 전달
</평가 기준>

<Rubric>
0.0 ~ 1.0 사이 점수를 부여하세요:
- 0.0에 가까울수록: 답변 A가 전반적으로 우수
- 0.5: 두 답변이 동등
- 1.0에 가까울수록: 답변 B가 전반적으로 우수
</Rubric>
"""

# 연속 점수 비교 평가자 (continuous=True)
pairwise_score_evaluator = create_llm_as_judge(
    prompt=PAIRWISE_SCORE_PROMPT,
    feedback_key="pairwise_score",
    continuous=True,    # 0~1 연속 점수
    model="openai:gpt-4.1-mini",
)

# 평가 실행
result = pairwise_score_evaluator(
    input=question,
    prediction=gpt_response["answer"],
    prediction_b=gemini_response["answer"],
    reference=ground_truth,
)

print(f"[A] GPT 답변: {gpt_response['answer']}")
print(f"[B] Gemini 답변: {gemini_response['answer']}")
print("-" * 100)
score = result.get('score')
if score < 0.4:
    label = "A 우세"
elif score > 0.6:
    label = "B 우세"
else:
    label = "대등"
print(f"연속 점수: {score:.2f} → {label}")
print(f"평가 근거: {result.get('comment')}")

[A] GPT 답변: Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도하여 회장 겸 최대 주주가 되었고, 2008년에는 CEO로 취임했습니다. 그는 경영 변화 과정에서 2007년과 2008년에 기존 CEO들이 물러나도록 했으며, 2009년에는 공동 설립자 Martin Eberhard가 Musk를 상대로 소송을 제기했으나 기각되었습니다. 이 과정에서 Musk는 경영권 인수와 관련된 논란에 직면했습니다.
[B] Gemini 답변: Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도하여 회장 겸 최대 주주가 되었습니다. 2008년 10월에는 CEO가 되었으며, 이 과정에서 공동 설립자인 Martin Eberhard는 Musk를 상대로 소송을 제기했지만 나중에 기각되었습니다.
----------------------------------------------------------------------------------------------------
연속 점수: 0.20 → A 우세
평가 근거: 답변 A와 답변 B 모두 Tesla의 초기 자금 조달에서 Elon Musk의 역할과 CEO 취임 및 경영 변화에 대해 언급하고 있습니다. 답변 A는 2007년과 2008년에 기존 CEO들이 물러난 점과 2009년 Martin Eberhard의 소송 제기 및 기각 사실을 언급하여 경영 변화 과정과 관련된 논란을 좀 더 구체적으로 다루고 있습니다. 반면 답변 B는 이 부분을 간략히 언급하며 정보는 덜 상세하지만 핵심은 포함하고 있습니다. 그러나 참조 답변에서 언급된 Tesla가 직면한 내부 고발자 보복, 근로자 권리 침해, 안전 결함, 홍보 부족, Musk의 논란의 여지가 있는 발언과 관련된 소송, 정부 조사 및 비판 등은 두 답변 모두 포함하지 않아 완전성 측면에서는 제한적입니다. 간결성 면에서는 답변 B가 불필요한 내용 없이 핵심을 전달하는 반면, 답변 A는 조금 더 상세하고 길게 설명합니다. 종합적으로 보면, 답변 A가 참조 답변에서 제시된

`(1) 다차원 점수 평가`

- 여러 평가 기준에 대해 개별 점수를 부여
- 각 항목별 강점/약점 파악 가능


In [40]:
# 다차원 Pairwise 평가 - 항목별 개별 평가기 사용

# 각 차원별 평가 프롬프트 템플릿
def make_dimension_prompt(dimension_name, dimension_desc):
    return f"""두 AI 답변의 '{dimension_name}'을 비교하세요.

<평가 기준>
{dimension_desc}
</평가 기준>

<질문>
{{input}}
</질문>

<참조 답변>
{{reference}}
</참조 답변>

<답변 A>
{{prediction}}
</답변 A>

<답변 B>
{{prediction_b}}
</답변 B>

<Rubric>
- 0: 답변 A가 이 기준에서 더 우수
- 0.5: 동등
- 1: 답변 B가 이 기준에서 더 우수
</Rubric>
"""

# 평가 차원 정의
dimensions = {
    "accuracy": ("정확성", "사실 관계가 정확하고 참조 답변과 일치하는가?"),
    "relevance": ("관련성", "질문에 적절히 답변하고 핵심을 다루는가?"),
    "conciseness": ("간결성", "불필요한 내용 없이 핵심만 전달하는가?"),
    "helpfulness": ("유용성", "실질적으로 도움이 되는 정보를 제공하는가?"),
}

# 차원별 평가 실행
print("=== 다차원 Pairwise 평가 결과 ===\n")
dim_results = {}
for key, (name, desc) in dimensions.items():
    dim_evaluator = create_llm_as_judge(
        prompt=make_dimension_prompt(name, desc),
        feedback_key=f"pairwise_{key}",
        choices=[0.0, 0.5, 1.0],
        model="openai:gpt-4.1-mini",
    )
    result = dim_evaluator(
        input=question,
        prediction=gpt_response["answer"],
        prediction_b=gemini_response["answer"],
        reference=ground_truth,
    )
    dim_results[key] = result['score']
    print(f"  {name}: {result['score']} → {interpret_pairwise(result['score'])}")

# 종합 점수 계산
avg_score = sum(dim_results.values()) / len(dim_results)
print(f"\n종합 평균: {avg_score:.2f} → ", end="")
if avg_score < 0.4:
    print("A 우세 (gpt-4.1-mini)")
elif avg_score > 0.6:
    print("B 우세 (Gemini-1.5-flash)")
else:
    print("대등")

=== 다차원 Pairwise 평가 결과 ===

  정확성: 0.0 → A 승 (Response A is better)
  관련성: 0.0 → A 승 (Response A is better)
  간결성: 1.0 → B 승 (Response B is better)
  유용성: 0 → A 승 (Response A is better)

종합 평균: 0.25 → A 우세 (gpt-4.1-mini)


### **[실습]**

- 테스트셋(df_qa_test)에서 하나의 샘플을 선택합니다.
- 이 샘플에 대한 RAG 답변을 두 가지 모델로부터 구합니다.
- 두 가지 실행결과에 대한 A/B 테스트 분석을 수행합니다. (langfuse UI 확인)

In [ ]:
# 여기에 코드를 작성하세요.

---

## [실습] **RAG 성능 A/B 테스트**

- **OpenEvals**를 사용하여 RAG 답변의 품질을 평가합니다.

- 다음과 같은 **사용자 정의 평가 기준**을 정의하여 평가합니다. (예시)
    - Conciseness (간결성): 불필요한 반복이나 장황함 없이 핵심 내용 전달
    - Helpfulness (유용성): 실질적인 도움이 되는 정도
    - Harmfulness/Maliciousness (유해성): 해로운 내용 포함 여부

- 요구 사항:
    - 올라마(Ollama)에서 다운로드한 오픈소스 모델 성능을 gpt-4.1-mini 모델의 성능과 비교
    - 평가자 모델은 gpt-4.1 사용
    - 사용자 정의 프롬프트 사용
    - df_qa_test 전체 테스트셋에 대해서 평가를 수행
    - Reference-free 평가와 Reference-based 평가를 각각 수행 (1개 이상)


In [ ]:
# 여기에 코드를 작성하세요.